<a href="https://colab.research.google.com/github/ibrahimbarghout/robust-ecg-domain-generalization/blob/main/notebooks/09_ECG_ML_Generalization_Robustness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wfdb -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.3 MB/s eta 0:00:00


In [2]:
# ======================================================================
# STEP 09A.1 — MOUNT GOOGLE DRIVE
# ======================================================================

from google.colab import drive

drive.mount("/content/drive")

print()
print("=" * 70)
print("GOOGLE DRIVE MOUNT COMPLETE")
print("=" * 70)

Mounted at /content/drive

GOOGLE DRIVE MOUNT COMPLETE


In [3]:
# ======================================================================
# NOTEBOOK 09 — ECG ML GENERALIZATION & ROBUSTNESS
# STEP 09A — ENVIRONMENT & PROJECT VERIFICATION
# ======================================================================

import os
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import wfdb

print("=" * 70)
print("NOTEBOOK 09 — ECG ML GENERALIZATION & ROBUSTNESS")
print("=" * 70)

print()
print("PYTHON / LIBRARY ENVIRONMENT")
print("-" * 70)

print(f"Python:       {sys.version.split()[0]}")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"SciPy:        {scipy.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"PyTorch:      {torch.__version__}")
print(f"WFDB:         {wfdb.__version__}")

print()
print("DEVICE")
print("-" * 70)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

print()
print("PROJECT PATHS")
print("-" * 70)

PROJECT_PATH = "/content/drive/MyDrive/PTB-XL Research Project"

DATA_PATH = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

RESULTS_PATH = (
    "/content/drive/MyDrive/PTB-XL Research Project/results"
)

SPLIT_PATH = os.path.join(
    RESULTS_PATH,
    "ptbxl_patient_independent_split.csv"
)

TARGETS_PATH = os.path.join(
    RESULTS_PATH,
    "ptbxl_diagnostic_targets.csv"
)

ADAPTIVE_RESULTS_PATH = os.path.join(
    RESULTS_PATH,
    "heldout_adaptive_filtering.csv"
)

print(f"PROJECT_PATH:          {PROJECT_PATH}")
print(f"DATA_PATH:             {DATA_PATH}")
print(f"RESULTS_PATH:          {RESULTS_PATH}")
print(f"SPLIT_PATH:            {SPLIT_PATH}")
print(f"TARGETS_PATH:          {TARGETS_PATH}")

print()
print("PATH CHECK")
print("-" * 70)

paths = {
    "Project": PROJECT_PATH,
    "Dataset": DATA_PATH,
    "Results": RESULTS_PATH,
    "Patient-independent split": SPLIT_PATH,
    "Diagnostic targets": TARGETS_PATH,
    "Adaptive filtering results": ADAPTIVE_RESULTS_PATH,
}

all_paths_ok = True

for name, path in paths.items():
    exists = os.path.exists(path)

    if exists:
        print(f"[PASS] {name}")
    else:
        print(f"[FAIL] {name} — {path}")
        all_paths_ok = False

print()
print("=" * 70)

if all_paths_ok:
    print("STEP 09A STATUS: PASS")
    print("Environment and required project paths verified.")
else:
    print("STEP 09A STATUS: CHECK REQUIRED PATHS")

print("=" * 70)

NOTEBOOK 09 — ECG ML GENERALIZATION & ROBUSTNESS

PYTHON / LIBRARY ENVIRONMENT
----------------------------------------------------------------------
Python:       3.13.15
NumPy:        2.1.3
Pandas:       2.2.3
SciPy:        1.16.3
Scikit-learn: 1.6.1
PyTorch:      2.11.0+cpu
WFDB:         4.3.1

DEVICE
----------------------------------------------------------------------
Device: cpu

PROJECT PATHS
----------------------------------------------------------------------
PROJECT_PATH:          /content/drive/MyDrive/PTB-XL Research Project
DATA_PATH:             /content/drive/MyDrive/PTB-XL Research Project/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3
RESULTS_PATH:          /content/drive/MyDrive/PTB-XL Research Project/results
SPLIT_PATH:            /content/drive/MyDrive/PTB-XL Research Project/results/ptbxl_patient_independent_split.csv
TARGETS_PATH:          /content/drive/MyDrive/PTB-XL Research Project/results/ptbxl_diagnostic_targets.csv

PATH CHECK
-

In [4]:
# ======================================================================
# STEP 09B — VERIFY PATIENT-INDEPENDENT SPLIT & DIAGNOSTIC TARGETS
# ======================================================================

print("=" * 70)
print("STEP 09B — SPLIT & TARGET VERIFICATION")
print("=" * 70)

# ----------------------------------------------------------------------
# Load existing split and target files
# ----------------------------------------------------------------------

split_df = pd.read_csv(SPLIT_PATH)
targets_df = pd.read_csv(TARGETS_PATH)

print()
print("DATASET FILES")
print("-" * 70)

print(f"Split rows:   {len(split_df):,}")
print(f"Target rows:  {len(targets_df):,}")

print()
print("SPLIT COLUMNS")
print("-" * 70)

print(split_df.columns.tolist())

print()
print("TARGET COLUMNS")
print("-" * 70)

print(targets_df.columns.tolist())

# ----------------------------------------------------------------------
# Identify split column
# ----------------------------------------------------------------------

possible_split_columns = [
    "split",
    "Split",
    "dataset_split",
    "Set"
]

split_column = None

for col in possible_split_columns:
    if col in split_df.columns:
        split_column = col
        break

if split_column is None:
    raise ValueError(
        "Could not identify the split column. "
        f"Available columns: {split_df.columns.tolist()}"
    )

print()
print(f"Detected split column: {split_column}")

print()
print("SPLIT DISTRIBUTION")
print("-" * 70)

print(
    split_df[split_column]
    .value_counts(dropna=False)
    .sort_index()
)

# ----------------------------------------------------------------------
# Identify ECG ID columns
# ----------------------------------------------------------------------

if "ecg_id" in split_df.columns:
    split_id_column = "ecg_id"
elif "ECG_ID" in split_df.columns:
    split_id_column = "ECG_ID"
else:
    raise ValueError(
        "Could not identify ECG ID column in split file."
    )

if "ecg_id" in targets_df.columns:
    target_id_column = "ecg_id"
elif "ECG_ID" in targets_df.columns:
    target_id_column = "ECG_ID"
else:
    raise ValueError(
        "Could not identify ECG ID column in target file."
    )

# ----------------------------------------------------------------------
# Identify target columns
# ----------------------------------------------------------------------

expected_targets = [
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
]

missing_targets = [
    target for target in expected_targets
    if target not in targets_df.columns
]

if missing_targets:
    raise ValueError(
        "Missing expected target columns: "
        f"{missing_targets}"
    )

# ----------------------------------------------------------------------
# Basic consistency checks
# ----------------------------------------------------------------------

split_ids = set(
    split_df[split_id_column].astype(int)
)

target_ids = set(
    targets_df[target_id_column].astype(int)
)

print()
print("ECG ID CONSISTENCY")
print("-" * 70)

print(f"ECGs in split:       {len(split_ids):,}")
print(f"ECGs in targets:     {len(target_ids):,}")
print(f"Split IDs missing targets: {len(split_ids - target_ids):,}")
print(f"Target IDs missing split:   {len(target_ids - split_ids):,}")

if split_ids == target_ids:
    print("[PASS] Split and target files contain identical ECG IDs.")
else:
    print("[FAIL] ECG ID mismatch detected.")

# ----------------------------------------------------------------------
# Build merged verification table
# ----------------------------------------------------------------------

verification_df = split_df.merge(
    targets_df,
    left_on=split_id_column,
    right_on=target_id_column,
    how="inner"
)

print()
print("MERGED DATASET")
print("-" * 70)

print(f"Merged rows: {len(verification_df):,}")

# ----------------------------------------------------------------------
# Per-split target prevalence
# ----------------------------------------------------------------------

print()
print("TARGET PREVALENCE BY SPLIT")
print("-" * 70)

for split_name in sorted(
    verification_df[split_column].dropna().unique()
):

    subset = verification_df[
        verification_df[split_column] == split_name
    ]

    print()
    print(f"{split_name}: {len(subset):,} ECGs")

    for target in expected_targets:
        prevalence = subset[target].mean() * 100
        positives = int(subset[target].sum())

        print(
            f"  {target:5s}: "
            f"{positives:5d} positive "
            f"({prevalence:6.2f}%)"
        )

# ----------------------------------------------------------------------
# Patient overlap verification
# ----------------------------------------------------------------------

print()
print("PATIENT INDEPENDENCE")
print("-" * 70)

# The split file should contain patient_id.
if "patient_id" in split_df.columns:

    patient_sets = {}

    for split_name in sorted(
        split_df[split_column].dropna().unique()
    ):
        patient_sets[split_name] = set(
            split_df.loc[
                split_df[split_column] == split_name,
                "patient_id"
            ].astype(int)
        )

    split_names = list(patient_sets.keys())

    total_overlap = 0

    for i in range(len(split_names)):
        for j in range(i + 1, len(split_names)):

            a = split_names[i]
            b = split_names[j]

            overlap = patient_sets[a].intersection(
                patient_sets[b]
            )

            print(
                f"{a} ↔ {b}: "
                f"{len(overlap):,} overlapping patients"
            )

            total_overlap += len(overlap)

    if total_overlap == 0:
        print()
        print(
            "[PASS] No patient appears in more than one split."
        )
    else:
        print()
        print(
            "[FAIL] Patient overlap detected."
        )

else:
    print(
        "[WARNING] patient_id not found in split file; "
        "patient independence could not be rechecked here."
    )

# ----------------------------------------------------------------------
# Final verification
# ----------------------------------------------------------------------

print()
print("=" * 70)

ids_ok = split_ids == target_ids
rows_ok = len(verification_df) == len(split_df)

if ids_ok and rows_ok:
    print("STEP 09B STATUS: PASS")
    print()
    print("Existing patient-independent split and diagnostic targets")
    print("are ready for the ML generalization experiments.")
else:
    print("STEP 09B STATUS: CHECK REQUIRED")

print("=" * 70)

STEP 09B — SPLIT & TARGET VERIFICATION

DATASET FILES
----------------------------------------------------------------------
Split rows:   21,799
Target rows:  21,799

SPLIT COLUMNS
----------------------------------------------------------------------
['ecg_id', 'patient_id', 'strat_fold', 'split']

TARGET COLUMNS
----------------------------------------------------------------------
['ecg_id', 'NORM', 'MI', 'STTC', 'CD', 'HYP', 'ABNORMAL', 'REFERENCE_NORMAL']

Detected split column: split

SPLIT DISTRIBUTION
----------------------------------------------------------------------
split
test           2198
train         17418
validation     2183
Name: count, dtype: int64

ECG ID CONSISTENCY
----------------------------------------------------------------------
ECGs in split:       21,799
ECGs in targets:     21,799
Split IDs missing targets: 0
Target IDs missing split:   0
[PASS] Split and target files contain identical ECG IDs.

MERGED DATASET
------------------------------------------

In [5]:
# ======================================================================
# STEP 09C — INSPECT EXISTING SIGNAL / PREPROCESSING ARTIFACTS
# ======================================================================

print("=" * 70)
print("STEP 09C — EXISTING SIGNAL & PREPROCESSING ARTIFACTS")
print("=" * 70)

# ----------------------------------------------------------------------
# Existing result files relevant to Notebook 09
# ----------------------------------------------------------------------

print()
print("RELEVANT RESULT FILES")
print("-" * 70)

relevant_files = [
    "ptbxl_patient_independent_split.csv",
    "ptbxl_diagnostic_targets.csv",
    "heldout_adaptive_filtering.csv",
    "heldout_adaptive_morphology.csv",
    "heldout_adaptive_timing_amplitude.csv",
    "step46_final_statistics_summary.csv",
    "step47e_final_paper_results_table.csv",
]

for filename in relevant_files:

    path = os.path.join(
        RESULTS_PATH,
        filename
    )

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"[FOUND] {filename:<45} {size_mb:8.2f} MB")
    else:
        print(f"[MISSING] {filename}")

# ----------------------------------------------------------------------
# Inspect existing filtering functions in the current runtime
# ----------------------------------------------------------------------

print()
print("EXISTING FILTER FUNCTIONS")
print("-" * 70)

filter_function_names = [
    "bandpass_filter",
    "apply_bandpass",
    "compute_baseline",
    "baseline_suppression_percent",
    "morphology_correlation",
]

for name in filter_function_names:

    if name in globals():
        obj = globals()[name]

        print(f"[FOUND] {name}")
        print(f"        {type(obj).__name__}")

    else:
        print(f"[NOT LOADED] {name}")

# ----------------------------------------------------------------------
# Inspect dataset structure
# ----------------------------------------------------------------------

print()
print("PTB-XL DATASET STRUCTURE")
print("-" * 70)

metadata_path = os.path.join(
    DATA_PATH,
    "ptbxl_database.csv"
)

if os.path.exists(metadata_path):

    metadata_09 = pd.read_csv(
        metadata_path,
        index_col="ecg_id"
    )

    print(f"Metadata shape: {metadata_09.shape}")
    print(
        f"Metadata columns: "
        f"{len(metadata_09.columns)}"
    )

    print()
    print("Required signal-path column:")

    if "filename_hr" in metadata_09.columns:
        print("[PASS] filename_hr")
    else:
        print("[FAIL] filename_hr missing")

else:

    metadata_09 = None

    print(
        "[FAIL] PTB-XL metadata file not found:"
    )
    print(metadata_path)

# ----------------------------------------------------------------------
# Inspect one actual high-resolution record
# ----------------------------------------------------------------------

print()
print("HIGH-RESOLUTION ECG RECORD CHECK")
print("-" * 70)

if metadata_09 is not None and "filename_hr" in metadata_09.columns:

    example_ecg_id = int(metadata_09.index[0])

    relative_record = metadata_09.loc[
        example_ecg_id,
        "filename_hr"
    ]

    record_path = os.path.join(
        DATA_PATH,
        relative_record
    )

    print(f"Example ECG ID: {example_ecg_id}")
    print(f"Record path:    {record_path}")

    if os.path.exists(record_path + ".dat"):

        record = wfdb.rdrecord(
            record_path
        )

        print()
        print(f"Signal shape:   {record.p_signal.shape}")
        print(f"Sampling rate:  {record.fs} Hz")
        print(f"Number of leads: {record.p_signal.shape[1]}")
        print(
            f"Lead names:     "
            f"{record.sig_name}"
        )

        print()
        print("[PASS] High-resolution ECG record readable.")

    else:

        print(
            "[FAIL] Record file not found."
        )

# ----------------------------------------------------------------------
# Final status
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("STEP 09C STATUS: INSPECTION COMPLETE")
print("=" * 70)

STEP 09C — EXISTING SIGNAL & PREPROCESSING ARTIFACTS

RELEVANT RESULT FILES
----------------------------------------------------------------------
[FOUND] ptbxl_patient_independent_split.csv               0.45 MB
[FOUND] ptbxl_diagnostic_targets.csv                      0.41 MB
[FOUND] heldout_adaptive_filtering.csv                    3.67 MB
[FOUND] heldout_adaptive_morphology.csv                   2.93 MB
[FOUND] heldout_adaptive_timing_amplitude.csv             5.70 MB
[FOUND] step46_final_statistics_summary.csv               0.00 MB
[FOUND] step47e_final_paper_results_table.csv             0.00 MB

EXISTING FILTER FUNCTIONS
----------------------------------------------------------------------
[NOT LOADED] bandpass_filter
[NOT LOADED] apply_bandpass
[NOT LOADED] compute_baseline
[NOT LOADED] baseline_suppression_percent
[NOT LOADED] morphology_correlation

PTB-XL DATASET STRUCTURE
----------------------------------------------------------------------
Metadata shape: (21799, 27)
Met

In [6]:
# ======================================================================
# STEP 09D — BASELINE FEATURE EXTRACTION TEST
# ======================================================================

from scipy import signal, stats
from scipy.fft import rfft, rfftfreq

print("=" * 70)
print("STEP 09D — BASELINE FEATURE EXTRACTION TEST")
print("=" * 70)

# ----------------------------------------------------------------------
# Frozen preprocessing definition from Notebook 08
# ----------------------------------------------------------------------

def bandpass_filter_09(
    ecg_signal,
    fs=500,
    lowcut=0.5,
    highcut=40.0,
    order=4
):
    """
    Frozen preprocessing reference from Notebook 08.
    """
    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = signal.butter(
        order,
        [low, high],
        btype="bandpass"
    )

    return signal.filtfilt(
        b,
        a,
        ecg_signal,
        axis=0
    )


# ----------------------------------------------------------------------
# Feature extraction
# ----------------------------------------------------------------------

def spectral_entropy_09(x, fs):
    """
    Normalized spectral entropy.
    """
    spectrum = np.abs(rfft(x)) ** 2

    if np.sum(spectrum) <= 1e-12:
        return 0.0

    power = spectrum / np.sum(spectrum)

    entropy = -np.sum(
        power * np.log2(power + 1e-12)
    )

    max_entropy = np.log2(len(power))

    if max_entropy <= 0:
        return 0.0

    return entropy / max_entropy


def dominant_frequency_09(x, fs):
    """
    Dominant non-zero frequency component.
    """
    spectrum = np.abs(rfft(x)) ** 2
    frequencies = rfftfreq(len(x), d=1.0 / fs)

    if len(spectrum) <= 1:
        return 0.0

    spectrum[0] = 0.0

    return float(
        frequencies[np.argmax(spectrum)]
    )


def extract_lead_features_09(x, fs):
    """
    Extract compact time- and frequency-domain features
    from one ECG lead.
    """

    x = np.asarray(x, dtype=np.float64)

    return {
        "rms": np.sqrt(np.mean(x ** 2)),
        "std": np.std(x),
        "ptp": np.ptp(x),
        "mean_abs": np.mean(np.abs(x)),
        "max_abs": np.max(np.abs(x)),
        "range": np.max(x) - np.min(x),
        "skewness": stats.skew(x),
        "kurtosis": stats.kurtosis(x),
        "dominant_frequency": dominant_frequency_09(
            x,
            fs
        ),
        "spectral_entropy": spectral_entropy_09(
            x,
            fs
        ),
    }


# ----------------------------------------------------------------------
# Load one ECG
# ----------------------------------------------------------------------

example_ecg_id = 1

relative_record = metadata_09.loc[
    example_ecg_id,
    "filename_hr"
]

record_path_09 = os.path.join(
    DATA_PATH,
    relative_record
)

record_09 = wfdb.rdrecord(
    record_path_09
)

raw_ecg_09 = record_09.p_signal
fs_09 = float(record_09.fs)

# Frozen fixed preprocessing
filtered_ecg_09 = bandpass_filter_09(
    raw_ecg_09,
    fs=fs_09,
    lowcut=0.5,
    highcut=40.0,
    order=4
)

# ----------------------------------------------------------------------
# Extract features from all 12 leads
# ----------------------------------------------------------------------

feature_row_09 = {}

for lead_idx, lead_name in enumerate(
    record_09.sig_name
):

    lead_features = extract_lead_features_09(
        filtered_ecg_09[:, lead_idx],
        fs_09
    )

    for feature_name, value in lead_features.items():

        feature_row_09[
            f"{lead_name}_{feature_name}"
        ] = value

feature_test_df_09 = pd.DataFrame(
    [feature_row_09]
)

# ----------------------------------------------------------------------
# Display results
# ----------------------------------------------------------------------

print()
print("ECG")
print("-" * 70)

print(f"ECG ID:       {example_ecg_id}")
print(f"Sampling rate: {fs_09} Hz")
print(f"Signal shape:  {raw_ecg_09.shape}")
print(f"Features:      {feature_test_df_09.shape[1]}")

print()
print("FEATURES PER LEAD")
print("-" * 70)

print(
    feature_test_df_09.columns.tolist()[:10]
)

print()
print("FEATURE VALUE CHECK")
print("-" * 70)

feature_values = feature_test_df_09.iloc[0].values

print(
    f"Finite values: "
    f"{np.isfinite(feature_values).sum()}/"
    f"{len(feature_values)}"
)

print(
    f"Minimum feature value: "
    f"{np.nanmin(feature_values):.6f}"
)

print(
    f"Maximum feature value: "
    f"{np.nanmax(feature_values):.6f}"
)

print()
print("FIRST 10 FEATURES")
print("-" * 70)

print(
    feature_test_df_09.iloc[
        0,
        :10
    ].to_string()
)

# ----------------------------------------------------------------------
# Final status
# ----------------------------------------------------------------------

print()
print("=" * 70)

if (
    feature_test_df_09.shape[1] == 12 * 10
    and
    np.isfinite(feature_values).all()
):
    print("STEP 09D STATUS: PASS")
    print()
    print(
        "The baseline feature representation successfully "
        "extracts 10 features from each of the 12 ECG leads."
    )
else:
    print("STEP 09D STATUS: CHECK FEATURE EXTRACTION")

print("=" * 70)

STEP 09D — BASELINE FEATURE EXTRACTION TEST

ECG
----------------------------------------------------------------------
ECG ID:       1
Sampling rate: 500.0 Hz
Signal shape:  (5000, 12)
Features:      120

FEATURES PER LEAD
----------------------------------------------------------------------
['I_rms', 'I_std', 'I_ptp', 'I_mean_abs', 'I_max_abs', 'I_range', 'I_skewness', 'I_kurtosis', 'I_dominant_frequency', 'I_spectral_entropy']

FEATURE VALUE CHECK
----------------------------------------------------------------------
Finite values: 120/120
Minimum feature value: -4.193955
Maximum feature value: 23.557859

FIRST 10 FEATURES
----------------------------------------------------------------------
I_rms                    0.093856
I_std                    0.093854
I_ptp                    0.730796
I_mean_abs               0.055154
I_max_abs                0.620402
I_range                  0.730796
I_skewness               3.676936
I_kurtosis              17.182664
I_dominant_frequency  

In [7]:
# ============================================================
# STEP 09E-FIX — RELOAD EXISTING SPLIT + TARGET ARTIFACTS
# ============================================================

import os
import pandas as pd

print("=" * 70)
print("STEP 09E-FIX — RELOADING EXISTING ARTIFACTS")
print("=" * 70)

# ------------------------------------------------------------
# Existing frozen artifact paths
# ------------------------------------------------------------

SPLIT_PATH = os.path.join(
    RESULTS_PATH,
    "ptbxl_patient_independent_split.csv"
)

TARGETS_PATH = os.path.join(
    RESULTS_PATH,
    "ptbxl_diagnostic_targets.csv"
)

# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------

if not os.path.exists(SPLIT_PATH):
    raise FileNotFoundError(
        f"Split file not found:\n{SPLIT_PATH}"
    )

if not os.path.exists(TARGETS_PATH):
    raise FileNotFoundError(
        f"Target file not found:\n{TARGETS_PATH}"
    )

print("\nArtifact files found: PASS")

# ------------------------------------------------------------
# Load frozen artifacts
# ------------------------------------------------------------

split_09 = pd.read_csv(
    SPLIT_PATH
)

targets_09 = pd.read_csv(
    TARGETS_PATH
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nSplit shape:")
print(split_09.shape)

print("\nTarget shape:")
print(targets_09.shape)

print("\nSplit distribution:")
print(
    split_09["split"]
    .value_counts()
    .sort_index()
)

print("\nTarget columns:")
print(
    targets_09.columns.tolist()
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

if len(split_09) != 21799:
    raise RuntimeError(
        f"Unexpected split rows: {len(split_09)}"
    )

if len(targets_09) != 21799:
    raise RuntimeError(
        f"Unexpected target rows: {len(targets_09)}"
    )

if not split_09["ecg_id"].is_unique:
    raise RuntimeError(
        "Duplicate ECG IDs in split artifact."
    )

if not targets_09["ecg_id"].is_unique:
    raise RuntimeError(
        "Duplicate ECG IDs in target artifact."
    )

if set(split_09["ecg_id"]) != set(targets_09["ecg_id"]):
    raise RuntimeError(
        "Split and target ECG IDs do not match."
    )

print("\nECG ID consistency: PASS")

print("\n" + "=" * 70)
print("ARTIFACT RELOAD STATUS: PASS")
print("=" * 70)
print("09B artifacts restored in the current runtime.")
print("No previous analysis was rerun.")

STEP 09E-FIX — RELOADING EXISTING ARTIFACTS

Artifact files found: PASS

Split shape:
(21799, 4)

Target shape:
(21799, 8)

Split distribution:
split
test           2198
train         17418
validation     2183
Name: count, dtype: int64

Target columns:
['ecg_id', 'NORM', 'MI', 'STTC', 'CD', 'HYP', 'ABNORMAL', 'REFERENCE_NORMAL']

ECG ID consistency: PASS

ARTIFACT RELOAD STATUS: PASS
09B artifacts restored in the current runtime.
No previous analysis was rerun.


In [8]:
# ============================================================
# STEP 09E — FULL BASELINE FEATURE EXTRACTION
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import wfdb

print("=" * 70)
print("STEP 09E — FULL BASELINE FEATURE EXTRACTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify required objects from previous steps
# ------------------------------------------------------------

required_objects = [
    "metadata_09",
    "split_09",
    "targets_09",
    "bandpass_filter_09",
    "extract_lead_features_09"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        f"Missing required objects: {missing_objects}"
    )

print("\nRequired objects: PASS")


# ------------------------------------------------------------
# 2. Define feature order
# ------------------------------------------------------------

lead_names_09 = [
    "I", "II", "III", "AVR", "AVL", "AVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

feature_names_09 = [
    "rms",
    "std",
    "ptp",
    "mean_abs",
    "max_abs",
    "range",
    "skewness",
    "kurtosis",
    "dominant_frequency",
    "spectral_entropy"
]

all_feature_names_09 = [
    f"{lead}_{feature}"
    for lead in lead_names_09
    for feature in feature_names_09
]

assert len(all_feature_names_09) == 120

print(f"Features per ECG: {len(all_feature_names_09)}")


# ------------------------------------------------------------
# 3. Merge split + targets + metadata
# ------------------------------------------------------------

dataset_09 = (
    split_09[
        ["ecg_id", "patient_id", "split"]
    ]
    .merge(
        targets_09,
        on="ecg_id",
        how="inner"
    )
)

if len(dataset_09) != 21799:
    raise RuntimeError(
        f"Unexpected merged dataset size: {len(dataset_09)}"
    )

print(f"ECGs to process: {len(dataset_09):,}")


# ------------------------------------------------------------
# 4. Prepare output locations
# ------------------------------------------------------------

FEATURES_DIR_09 = os.path.join(
    RESULTS_PATH,
    "baseline_features"
)

os.makedirs(FEATURES_DIR_09, exist_ok=True)

FEATURES_CSV_PATH_09 = os.path.join(
    FEATURES_DIR_09,
    "ptbxl_baseline_features_120.csv"
)


# ------------------------------------------------------------
# 5. Avoid accidental recomputation
# ------------------------------------------------------------

if os.path.exists(FEATURES_CSV_PATH_09):

    print("\nExisting feature file detected:")
    print(FEATURES_CSV_PATH_09)

    existing_features_09 = pd.read_csv(
        FEATURES_CSV_PATH_09
    )

    print(
        f"Existing shape: "
        f"{existing_features_09.shape}"
    )

    if (
        len(existing_features_09) == 21799
        and len(
            set(all_feature_names_09)
            - set(existing_features_09.columns)
        ) == 0
    ):
        print("\nCached feature matrix is valid.")
        print("STEP 09E STATUS: PASS")
        print("Using existing cached features.")

    else:
        print(
            "\nExisting file does not match expected "
            "dimensions. Recomputing."
        )
        existing_features_09 = None

else:
    existing_features_09 = None


# ------------------------------------------------------------
# 6. Full extraction
# ------------------------------------------------------------

if existing_features_09 is None:

    start_time_09 = time.time()

    feature_rows_09 = []

    total_09 = len(dataset_09)

    for counter_09, (_, row_09) in enumerate(
        dataset_09.iterrows(),
        start=1
    ):

        ecg_id_09 = int(row_09["ecg_id"])

        record_path_09 = os.path.join(
            DATA_PATH,
            row_09["filename_hr"]
            if "filename_hr" in row_09
            else metadata_09.loc[
                ecg_id_09,
                "filename_hr"
            ]
        )

        # ----------------------------------------------------
        # Read ECG
        # ----------------------------------------------------

        record_09 = wfdb.rdrecord(
            record_path_09
        )

        signal_09 = np.asarray(
            record_09.p_signal,
            dtype=np.float64
        )

        fs_09 = float(
            record_09.fs
        )

        if signal_09.shape != (5000, 12):
            raise RuntimeError(
                f"Unexpected signal shape for ECG "
                f"{ecg_id_09}: {signal_09.shape}"
            )

        # ----------------------------------------------------
        # Apply frozen preprocessing
        # ----------------------------------------------------

        filtered_signal_09 = bandpass_filter_09(
            signal_09,
            fs=fs_09
        )

        # ----------------------------------------------------
        # Extract 10 features × 12 leads
        # ----------------------------------------------------

        row_features_09 = {
            "ecg_id": ecg_id_09
        }

        for lead_index_09, lead_name_09 in enumerate(
            lead_names_09
        ):

            lead_signal_09 = filtered_signal_09[
                :, lead_index_09
            ]

            lead_features_09 = extract_lead_features_09(
                lead_signal_09,
                fs_09
            )

            for feature_name_09 in feature_names_09:

                row_features_09[
                    f"{lead_name_09}_{feature_name_09}"
                ] = lead_features_09[
                    feature_name_09
                ]

        feature_rows_09.append(
            row_features_09
        )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            counter_09 == 1
            or counter_09 % 500 == 0
            or counter_09 == total_09
        ):

            elapsed_09 = time.time() - start_time_09

            rate_09 = (
                counter_09 / elapsed_09
                if elapsed_09 > 0
                else 0
            )

            remaining_09 = (
                (total_09 - counter_09) / rate_09
                if rate_09 > 0
                else 0
            )

            print(
                f"{counter_09:>6,}/{total_09:,} "
                f"| {rate_09:.2f} ECG/s "
                f"| ETA {remaining_09/60:.1f} min"
            )

    # --------------------------------------------------------
    # Convert to DataFrame
    # --------------------------------------------------------

    feature_matrix_09 = pd.DataFrame(
        feature_rows_09
    )

    expected_columns_09 = [
        "ecg_id"
    ] + all_feature_names_09

    feature_matrix_09 = feature_matrix_09[
        expected_columns_09
    ]

    # --------------------------------------------------------
    # Verify
    # --------------------------------------------------------

    expected_shape_09 = (
        21799,
        121
    )

    if feature_matrix_09.shape != expected_shape_09:
        raise RuntimeError(
            f"Unexpected feature matrix shape: "
            f"{feature_matrix_09.shape}; "
            f"expected {expected_shape_09}"
        )

    numeric_features_09 = feature_matrix_09[
        all_feature_names_09
    ]

    if not np.isfinite(
        numeric_features_09.to_numpy()
    ).all():
        raise RuntimeError(
            "Non-finite feature values detected."
        )

    if feature_matrix_09["ecg_id"].duplicated().any():
        raise RuntimeError(
            "Duplicate ECG IDs detected."
        )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    feature_matrix_09.to_csv(
        FEATURES_CSV_PATH_09,
        index=False
    )

    elapsed_total_09 = time.time() - start_time_09

    print("\nExtraction complete.")
    print(
        f"Elapsed time: "
        f"{elapsed_total_09/60:.2f} minutes"
    )

else:

    feature_matrix_09 = existing_features_09[
        ["ecg_id"] + all_feature_names_09
    ]


# ------------------------------------------------------------
# 7. Attach split information
# ------------------------------------------------------------

feature_matrix_09 = feature_matrix_09.merge(
    split_09[
        ["ecg_id", "patient_id", "split"]
    ],
    on="ecg_id",
    how="left"
)

# ------------------------------------------------------------
# 8. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL FEATURE MATRIX CHECK")
print("=" * 70)

print(
    f"Rows:              {len(feature_matrix_09):,}"
)

print(
    f"Feature columns:   {len(all_feature_names_09)}"
)

print(
    f"Total columns:     {feature_matrix_09.shape[1]}"
)

print("\nSplit distribution:")
print(
    feature_matrix_09["split"]
    .value_counts()
    .sort_index()
)

print("\nFinite values:")

finite_count_09 = np.isfinite(
    feature_matrix_09[
        all_feature_names_09
    ].to_numpy()
).sum()

total_values_09 = (
    len(feature_matrix_09)
    * len(all_feature_names_09)
)

print(
    f"{finite_count_09:,}/"
    f"{total_values_09:,}"
)

print("\nCached feature file:")
print(FEATURES_CSV_PATH_09)

print("\n" + "=" * 70)
print("STEP 09E STATUS: PASS")
print("=" * 70)

STEP 09E — FULL BASELINE FEATURE EXTRACTION

Required objects: PASS
Features per ECG: 120
ECGs to process: 21,799
     1/21,799 | 5.92 ECG/s | ETA 61.3 min
   500/21,799 | 1.60 ECG/s | ETA 222.5 min
 1,000/21,799 | 1.61 ECG/s | ETA 214.8 min
 1,500/21,799 | 1.61 ECG/s | ETA 210.6 min
 2,000/21,799 | 1.60 ECG/s | ETA 206.4 min
 2,500/21,799 | 1.57 ECG/s | ETA 205.5 min
 3,000/21,799 | 1.57 ECG/s | ETA 199.4 min
 3,500/21,799 | 1.57 ECG/s | ETA 194.3 min
 4,000/21,799 | 1.57 ECG/s | ETA 188.9 min
 4,500/21,799 | 1.57 ECG/s | ETA 183.2 min
 5,000/21,799 | 1.58 ECG/s | ETA 176.8 min
 5,500/21,799 | 1.60 ECG/s | ETA 169.9 min
 6,000/21,799 | 1.61 ECG/s | ETA 163.7 min
 6,500/21,799 | 1.61 ECG/s | ETA 157.9 min
 7,000/21,799 | 1.62 ECG/s | ETA 152.3 min
 7,500/21,799 | 1.63 ECG/s | ETA 146.2 min
 8,000/21,799 | 1.64 ECG/s | ETA 140.4 min
 8,500/21,799 | 1.64 ECG/s | ETA 135.1 min
 9,000/21,799 | 1.64 ECG/s | ETA 129.8 min
 9,500/21,799 | 1.65 ECG/s | ETA 124.2 min
10,000/21,799 | 1.65 ECG/s 

RuntimeError: Non-finite feature values detected.

In [9]:
# ============================================================
# STEP 09E-DIAGNOSTIC — IDENTIFY NON-FINITE FEATURES
# ============================================================

print("=" * 70)
print("STEP 09E-DIAGNOSTIC — NON-FINITE FEATURE INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Confirm the extracted feature matrix exists
# ------------------------------------------------------------

if "numeric_features_09" not in globals():
    raise RuntimeError(
        "numeric_features_09 is not available in the current runtime."
    )

print("\nFeature matrix shape:")
print(numeric_features_09.shape)

# ------------------------------------------------------------
# 2. Identify non-finite values by feature
# ------------------------------------------------------------

nonfinite_mask = ~np.isfinite(
    numeric_features_09.to_numpy(dtype=np.float64)
)

nonfinite_by_feature = pd.Series(
    nonfinite_mask.sum(axis=0),
    index=numeric_features_09.columns
)

nonfinite_by_feature = (
    nonfinite_by_feature[
        nonfinite_by_feature > 0
    ]
    .sort_values(ascending=False)
)

print("\nNon-finite values by feature:")

if len(nonfinite_by_feature) == 0:
    print("NONE")
else:
    print(nonfinite_by_feature)

# ------------------------------------------------------------
# 3. Identify affected ECG rows
# ------------------------------------------------------------

affected_rows = np.where(
    nonfinite_mask.any(axis=1)
)[0]

print("\nNumber of affected ECG rows:")
print(len(affected_rows))

# ------------------------------------------------------------
# 4. Show affected ECG IDs if available
# ------------------------------------------------------------

if "ecg_ids_09" in globals():

    affected_ecg_ids = [
        ecg_ids_09[i]
        for i in affected_rows
    ]

    print("\nAffected ECG IDs:")
    print(affected_ecg_ids[:50])

elif "feature_records_09" in globals():

    affected_ecg_ids = [
        feature_records_09[i]["ecg_id"]
        for i in affected_rows
    ]

    print("\nAffected ECG IDs:")
    print(affected_ecg_ids[:50])

else:
    print(
        "\nECG ID vector not found in memory; "
        "row indices are still available."
    )
    print("Affected row indices:")
    print(affected_rows[:50])

# ------------------------------------------------------------
# 5. Show actual problematic values
# ------------------------------------------------------------

if len(affected_rows) > 0:

    affected_feature_indices = np.where(
        nonfinite_mask.any(axis=0)
    )[0]

    print("\nProblematic feature values:")

    for row_idx in affected_rows[:10]:

        print(f"\nRow {row_idx}:")

        for col_idx in affected_feature_indices:

            value = numeric_features_09.iloc[
                row_idx,
                col_idx
            ]

            if not np.isfinite(value):
                print(
                    f"  {numeric_features_09.columns[col_idx]} "
                    f"= {value}"
                )

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

if len(nonfinite_by_feature) == 0:
    print("No non-finite feature values detected.")
else:
    print(
        f"Non-finite features: "
        f"{len(nonfinite_by_feature)}"
    )
    print(
        f"Affected ECG rows: "
        f"{len(affected_rows)}"
    )

print(
    "\nIMPORTANT: No values were modified and "
    "no feature extraction was rerun."
)

STEP 09E-DIAGNOSTIC — NON-FINITE FEATURE INSPECTION

Feature matrix shape:
(21799, 120)

Non-finite values by feature:
V5_skewness    1
V5_kurtosis    1
dtype: int64

Number of affected ECG rows:
1

ECG ID vector not found in memory; row indices are still available.
Affected row indices:
[12690]

Problematic feature values:

Row 12690:
  V5_skewness = nan
  V5_kurtosis = nan

DIAGNOSTIC COMPLETE
Non-finite features: 2
Affected ECG rows: 1

IMPORTANT: No values were modified and no feature extraction was rerun.


In [11]:
# ============================================================
# STEP 09E-DIAGNOSTIC-2 — INSPECT AFFECTED ECG / V5 SIGNAL
# CORRECTED: NO GLOBALS() ITERATION
# ============================================================

import os
import numpy as np
import pandas as pd
import wfdb

print("=" * 70)
print("STEP 09E-DIAGNOSTIC-2 — AFFECTED ECG SIGNAL INSPECTION")
print("=" * 70)

affected_row = 12690

# ------------------------------------------------------------
# 1. Recover ECG ID directly from the frozen split artifact
# ------------------------------------------------------------

split_check = pd.read_csv(SPLIT_PATH)

affected_ecg_id = int(
    split_check.iloc[affected_row]["ecg_id"]
)

print(f"\nAffected row index: {affected_row}")
print(f"Affected ECG ID: {affected_ecg_id}")

print(
    "Patient ID:",
    split_check.iloc[affected_row]["patient_id"]
)

print(
    "Split:",
    split_check.iloc[affected_row]["split"]
)

# ------------------------------------------------------------
# 2. Load PTB-XL metadata only for this ECG
# ------------------------------------------------------------

db_path = os.path.join(
    DATA_PATH,
    "ptbxl_database.csv"
)

db_09_diag = pd.read_csv(
    db_path,
    index_col="ecg_id"
)

filename_hr = db_09_diag.loc[
    affected_ecg_id,
    "filename_hr"
]

print("\nHigh-resolution record:")
print(filename_hr)

record_path = os.path.join(
    DATA_PATH,
    filename_hr
)

# ------------------------------------------------------------
# 3. Load affected ECG
# ------------------------------------------------------------

record = wfdb.rdrecord(record_path)

signal_raw = record.p_signal.astype(
    np.float64
)

lead_names = list(record.sig_name)

print("\nSignal shape:")
print(signal_raw.shape)

print("\nSampling frequency:")
print(record.fs)

print("\nLeads:")
print(lead_names)

# ------------------------------------------------------------
# 4. Inspect V5 raw signal
# ------------------------------------------------------------

if "V5" not in lead_names:
    raise RuntimeError("V5 lead not found.")

v5_index = lead_names.index("V5")

v5_raw = signal_raw[:, v5_index]

print("\nV5 RAW signal statistics:")
print(f"  Min:          {np.min(v5_raw):.12f}")
print(f"  Max:          {np.max(v5_raw):.12f}")
print(f"  Mean:         {np.mean(v5_raw):.12f}")
print(f"  Std:          {np.std(v5_raw):.12f}")
print(f"  Peak-to-peak: {np.ptp(v5_raw):.12f}")
print(f"  Finite:       {np.isfinite(v5_raw).all()}")

# ------------------------------------------------------------
# 5. Apply the SAME frozen preprocessing
# ------------------------------------------------------------

filtered_ecg = bandpass_filter_09(
    signal_raw,
    fs=record.fs,
    lowcut=0.5,
    highcut=40.0,
    order=4
)

v5_filtered = filtered_ecg[:, v5_index]

print("\nV5 FILTERED signal statistics:")
print(f"  Min:          {np.min(v5_filtered):.12f}")
print(f"  Max:          {np.max(v5_filtered):.12f}")
print(f"  Mean:         {np.mean(v5_filtered):.12f}")
print(f"  Std:          {np.std(v5_filtered):.12f}")
print(f"  Peak-to-peak: {np.ptp(v5_filtered):.12f}")
print(f"  Finite:       {np.isfinite(v5_filtered).all()}")

# ------------------------------------------------------------
# 6. Directly reproduce the failing features
# ------------------------------------------------------------

v5_skew = stats.skew(v5_filtered)
v5_kurtosis = stats.kurtosis(v5_filtered)

print("\nV5 statistical features:")
print(f"  Skewness: {v5_skew}")
print(f"  Kurtosis: {v5_kurtosis}")

# ------------------------------------------------------------
# 7. Check for numerical pathologies
# ------------------------------------------------------------

print("\nNumerical checks:")

print(
    "  Raw finite values:",
    np.isfinite(v5_raw).sum(),
    "/",
    len(v5_raw)
)

print(
    "  Filtered finite values:",
    np.isfinite(v5_filtered).sum(),
    "/",
    len(v5_filtered)
)

print(
    "  Unique raw values:",
    len(np.unique(v5_raw))
)

print(
    "  Unique filtered values:",
    len(np.unique(v5_filtered))
)

print(
    "  Filtered variance:",
    np.var(v5_filtered)
)

print("\n" + "=" * 70)
print("SIGNAL INSPECTION COMPLETE")
print("=" * 70)

STEP 09E-DIAGNOSTIC-2 — AFFECTED ECG SIGNAL INSPECTION

Affected row index: 12690
Affected ECG ID: 12722
Patient ID: 2818.0
Split: train

High-resolution record:
records500/12000/12722_hr

Signal shape:
(5000, 12)

Sampling frequency:
500

Leads:
['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

V5 RAW signal statistics:
  Min:          0.000000000000
  Max:          0.000000000000
  Mean:         0.000000000000
  Std:          0.000000000000
  Peak-to-peak: 0.000000000000
  Finite:       True

V5 FILTERED signal statistics:
  Min:          0.000000000000
  Max:          0.000000000000
  Mean:         0.000000000000
  Std:          0.000000000000
  Peak-to-peak: 0.000000000000
  Finite:       True

V5 statistical features:
  Skewness: nan
  Kurtosis: nan

Numerical checks:
  Raw finite values: 5000 / 5000
  Filtered finite values: 5000 / 5000
  Unique raw values: 1
  Unique filtered values: 1
  Filtered variance: 0.0

SIGNAL INSPECTION COMPLETE


In [12]:
# ============================================================
# STEP 09E-FIX — HANDLE CONSTANT-LEAD STATISTICAL FEATURES
# ============================================================

print("=" * 70)
print("STEP 09E-FIX — CONSTANT-LEAD FEATURE HANDLING")
print("=" * 70)

# ------------------------------------------------------------
# 1. Confirm the two expected problematic features
# ------------------------------------------------------------

expected_features = [
    "V5_skewness",
    "V5_kurtosis"
]

for feature in expected_features:

    if feature not in numeric_features_09.columns:
        raise RuntimeError(
            f"Expected feature not found: {feature}"
        )

# ------------------------------------------------------------
# 2. Confirm exactly one NaN remains in each feature
# ------------------------------------------------------------

for feature in expected_features:

    nan_count = numeric_features_09[
        feature
    ].isna().sum()

    if nan_count != 1:
        raise RuntimeError(
            f"Unexpected NaN count for {feature}: "
            f"{nan_count}"
        )

print("\nExpected NaN pattern confirmed: PASS")

# ------------------------------------------------------------
# 3. Replace ONLY the two mathematically undefined
#    constant-signal moments
# ------------------------------------------------------------

numeric_features_09.loc[
    numeric_features_09["V5_skewness"].isna(),
    "V5_skewness"
] = 0.0

numeric_features_09.loc[
    numeric_features_09["V5_kurtosis"].isna(),
    "V5_kurtosis"
] = 0.0

print(
    "\nV5_skewness NaN -> 0.0: PASS"
)

print(
    "V5_kurtosis NaN -> 0.0: PASS"
)

# ------------------------------------------------------------
# 4. Verify there are NO remaining non-finite values
# ------------------------------------------------------------

numeric_array_09 = numeric_features_09.to_numpy(
    dtype=np.float64
)

nonfinite_count_09 = np.sum(
    ~np.isfinite(numeric_array_09)
)

print("\nTotal non-finite feature values:")
print(nonfinite_count_09)

if nonfinite_count_09 != 0:
    raise RuntimeError(
        "Non-finite feature values still remain."
    )

print(
    "\nGlobal finite-value validation: PASS"
)

# ------------------------------------------------------------
# 5. Verify expected feature dimensions
# ------------------------------------------------------------

if numeric_features_09.shape != (21799, 120):
    raise RuntimeError(
        f"Unexpected feature matrix shape: "
        f"{numeric_features_09.shape}"
    )

print(
    "Feature matrix shape (21,799 × 120): PASS"
)

# ------------------------------------------------------------
# 6. Verify the patched values
# ------------------------------------------------------------

print("\nPatched values:")

print(
    "  V5_skewness:",
    numeric_features_09.iloc[
        12690
    ]["V5_skewness"]
)

print(
    "  V5_kurtosis:",
    numeric_features_09.iloc[
        12690
    ]["V5_kurtosis"]
)

print("\n" + "=" * 70)
print("STEP 09E-FIX STATUS: PASS")
print("=" * 70)
print(
    "Only the mathematically undefined moments of the "
    "constant V5 lead in ECG 12722 were replaced."
)
print(
    "No ECGs were removed and no feature extraction was rerun."
)

STEP 09E-FIX — CONSTANT-LEAD FEATURE HANDLING

Expected NaN pattern confirmed: PASS

V5_skewness NaN -> 0.0: PASS
V5_kurtosis NaN -> 0.0: PASS

Total non-finite feature values:
0

Global finite-value validation: PASS
Feature matrix shape (21,799 × 120): PASS

Patched values:
  V5_skewness: 0.0
  V5_kurtosis: 0.0

STEP 09E-FIX STATUS: PASS
Only the mathematically undefined moments of the constant V5 lead in ECG 12722 were replaced.
No ECGs were removed and no feature extraction was rerun.


/tmp/ipykernel_2577/3703126966.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  numeric_features_09.loc[
/tmp/ipykernel_2577/3703126966.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  numeric_features_09.loc[


In [13]:
# ============================================================
# STEP 09E-FINAL — FREEZE + SAVE BASELINE FEATURES
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 09E-FINAL — FREEZE + SAVE BASELINE FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Create an explicit independent copy
# ------------------------------------------------------------

numeric_features_09 = numeric_features_09.copy()

print("\nIndependent feature matrix copy: PASS")

# ------------------------------------------------------------
# 2. Attach ECG-level identifiers and split information
# ------------------------------------------------------------

feature_output_09 = numeric_features_09.copy()

feature_output_09.insert(
    0,
    "ecg_id",
    split_09["ecg_id"].to_numpy()
)

feature_output_09.insert(
    1,
    "patient_id",
    split_09["patient_id"].to_numpy()
)

feature_output_09.insert(
    2,
    "split",
    split_09["split"].to_numpy()
)

# ------------------------------------------------------------
# 3. Validate dimensions
# ------------------------------------------------------------

expected_rows = 21799
expected_features = 120
expected_columns = 123

print("\nOutput shape:")
print(feature_output_09.shape)

if feature_output_09.shape != (
    expected_rows,
    expected_columns
):
    raise RuntimeError(
        f"Unexpected output shape: "
        f"{feature_output_09.shape}"
    )

print(
    "Output dimensions "
    "(21,799 ECGs × 120 features + 3 metadata columns): PASS"
)

# ------------------------------------------------------------
# 4. Validate ECG IDs
# ------------------------------------------------------------

if not feature_output_09["ecg_id"].is_unique:
    raise RuntimeError(
        "Duplicate ECG IDs detected."
    )

if set(feature_output_09["ecg_id"]) != set(
    split_09["ecg_id"]
):
    raise RuntimeError(
        "Output ECG IDs do not match the frozen split."
    )

print("ECG ID integrity: PASS")

# ------------------------------------------------------------
# 5. Validate patient-independent split assignment
# ------------------------------------------------------------

split_counts = (
    feature_output_09["split"]
    .value_counts()
    .sort_index()
)

print("\nSplit distribution:")
print(split_counts)

expected_split_counts = {
    "train": 17418,
    "test": 2198,
    "validation": 2183
}

for split_name, expected_count in expected_split_counts.items():

    actual_count = int(
        split_counts.get(split_name, 0)
    )

    if actual_count != expected_count:
        raise RuntimeError(
            f"{split_name} count mismatch: "
            f"{actual_count} != {expected_count}"
        )

print("\nSplit integrity: PASS")

# ------------------------------------------------------------
# 6. Validate all feature values are finite
# ------------------------------------------------------------

feature_columns_09 = [
    col
    for col in numeric_features_09.columns
]

feature_array_09 = feature_output_09[
    feature_columns_09
].to_numpy(
    dtype=np.float64
)

nonfinite_total = np.sum(
    ~np.isfinite(feature_array_09)
)

print(
    "\nNon-finite feature values:",
    nonfinite_total
)

if nonfinite_total != 0:
    raise RuntimeError(
        "Non-finite feature values detected."
    )

print("Global finite-value validation: PASS")

# ------------------------------------------------------------
# 7. Confirm the previously affected ECG is now valid
# ------------------------------------------------------------

ecg_12722 = feature_output_09[
    feature_output_09["ecg_id"] == 12722
]

if len(ecg_12722) != 1:
    raise RuntimeError(
        "ECG 12722 not found exactly once."
    )

print("\nECG 12722 verification:")

print(
    "  V5_skewness:",
    float(ecg_12722["V5_skewness"].iloc[0])
)

print(
    "  V5_kurtosis:",
    float(ecg_12722["V5_kurtosis"].iloc[0])
)

# ------------------------------------------------------------
# 8. Save frozen feature artifact
# ------------------------------------------------------------

BASELINE_FEATURE_DIR = os.path.join(
    RESULTS_PATH,
    "baseline_features"
)

os.makedirs(
    BASELINE_FEATURE_DIR,
    exist_ok=True
)

BASELINE_FEATURE_PATH = os.path.join(
    BASELINE_FEATURE_DIR,
    "ptbxl_baseline_features_120.csv"
)

feature_output_09.to_csv(
    BASELINE_FEATURE_PATH,
    index=False
)

print("\nSaved feature artifact:")
print(BASELINE_FEATURE_PATH)

# ------------------------------------------------------------
# 9. Reload and verify the saved artifact
# ------------------------------------------------------------

reloaded_features_09 = pd.read_csv(
    BASELINE_FEATURE_PATH
)

print("\nReloaded artifact shape:")
print(reloaded_features_09.shape)

if reloaded_features_09.shape != (
    expected_rows,
    expected_columns
):
    raise RuntimeError(
        "Reloaded artifact has unexpected dimensions."
    )

reloaded_numeric_09 = reloaded_features_09[
    feature_columns_09
].to_numpy(
    dtype=np.float64
)

if not np.isfinite(
    reloaded_numeric_09
).all():
    raise RuntimeError(
        "Reloaded artifact contains non-finite values."
    )

if not reloaded_features_09[
    "ecg_id"
].is_unique:
    raise RuntimeError(
        "Reloaded artifact contains duplicate ECG IDs."
    )

print("Reloaded artifact validation: PASS")

# ------------------------------------------------------------
# 10. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 09E STATUS: COMPLETE / FROZEN")
print("=" * 70)

print(
    "21,799 ECGs retained."
)

print(
    "120 engineered features per ECG."
)

print(
    "0 non-finite feature values."
)

print(
    "Patient-independent split preserved."
)

print(
    "Baseline feature artifact saved and revalidated."
)

STEP 09E-FINAL — FREEZE + SAVE BASELINE FEATURES

Independent feature matrix copy: PASS

Output shape:
(21799, 123)
Output dimensions (21,799 ECGs × 120 features + 3 metadata columns): PASS
ECG ID integrity: PASS

Split distribution:
split
test           2198
train         17418
validation     2183
Name: count, dtype: int64

Split integrity: PASS

Non-finite feature values: 0
Global finite-value validation: PASS

ECG 12722 verification:
  V5_skewness: 0.0
  V5_kurtosis: 0.0

Saved feature artifact:
/content/drive/MyDrive/PTB-XL Research Project/results/baseline_features/ptbxl_baseline_features_120.csv

Reloaded artifact shape:
(21799, 123)
Reloaded artifact validation: PASS

STEP 09E STATUS: COMPLETE / FROZEN
21,799 ECGs retained.
120 engineered features per ECG.
0 non-finite feature values.
Patient-independent split preserved.
Baseline feature artifact saved and revalidated.


In [15]:
# ============================================================
# STEP 09F-FIX — MERGE FROZEN FEATURES + FROZEN TARGETS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

print("=" * 70)
print("STEP 09F-FIX — BASELINE ML DATASET CONSTRUCTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load the two frozen artifacts
# ------------------------------------------------------------

BASELINE_FEATURE_PATH = os.path.join(
    RESULTS_PATH,
    "baseline_features",
    "ptbxl_baseline_features_120.csv"
)

TARGETS_PATH = os.path.join(
    RESULTS_PATH,
    "ptbxl_diagnostic_targets.csv"
)

if not os.path.exists(BASELINE_FEATURE_PATH):
    raise FileNotFoundError(
        f"Feature artifact not found:\n"
        f"{BASELINE_FEATURE_PATH}"
    )

if not os.path.exists(TARGETS_PATH):
    raise FileNotFoundError(
        f"Target artifact not found:\n"
        f"{TARGETS_PATH}"
    )

features_09 = pd.read_csv(
    BASELINE_FEATURE_PATH
)

targets_09_ml = pd.read_csv(
    TARGETS_PATH
)

print("\nFeature artifact:")
print(features_09.shape)

print("\nTarget artifact:")
print(targets_09_ml.shape)

# ------------------------------------------------------------
# 2. Define locked diagnostic targets
# ------------------------------------------------------------

TARGET_COLUMNS_09 = [
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
]

# ------------------------------------------------------------
# 3. Validate source artifacts before merging
# ------------------------------------------------------------

if features_09.shape != (21799, 123):
    raise RuntimeError(
        f"Unexpected feature artifact shape: "
        f"{features_09.shape}"
    )

if targets_09_ml.shape[0] != 21799:
    raise RuntimeError(
        f"Unexpected target artifact rows: "
        f"{targets_09_ml.shape[0]}"
    )

if not features_09["ecg_id"].is_unique:
    raise RuntimeError(
        "Duplicate ECG IDs in feature artifact."
    )

if not targets_09_ml["ecg_id"].is_unique:
    raise RuntimeError(
        "Duplicate ECG IDs in target artifact."
    )

if not set(features_09["ecg_id"]) == set(
    targets_09_ml["ecg_id"]
):
    raise RuntimeError(
        "Feature and target ECG IDs do not match."
    )

print("\nSource artifact integrity: PASS")

# ------------------------------------------------------------
# 4. Merge ONLY on ECG ID
# ------------------------------------------------------------

ml_data_09 = features_09.merge(
    targets_09_ml[
        ["ecg_id"] + TARGET_COLUMNS_09
    ],
    on="ecg_id",
    how="inner",
    validate="one_to_one"
)

print("\nMerged ML dataset:")
print(ml_data_09.shape)

if ml_data_09.shape != (21799, 128):
    raise RuntimeError(
        f"Unexpected merged shape: "
        f"{ml_data_09.shape}"
    )

print(
    "21,799 ECGs × "
    "120 features + 3 metadata + 5 targets: PASS"
)

# ------------------------------------------------------------
# 5. Define feature columns
# ------------------------------------------------------------

FEATURE_COLUMNS_09 = [
    col
    for col in features_09.columns
    if col not in [
        "ecg_id",
        "patient_id",
        "split"
    ]
]

if len(FEATURE_COLUMNS_09) != 120:
    raise RuntimeError(
        f"Expected 120 features, found "
        f"{len(FEATURE_COLUMNS_09)}"
    )

print("Feature count: 120 PASS")
print("Target count: 5 PASS")

# ------------------------------------------------------------
# 6. Validate feature values
# ------------------------------------------------------------

X_all_09 = ml_data_09[
    FEATURE_COLUMNS_09
].to_numpy(
    dtype=np.float64
)

if not np.isfinite(X_all_09).all():
    raise RuntimeError(
        "Non-finite feature values detected."
    )

print("Feature finiteness: PASS")

# ------------------------------------------------------------
# 7. Construct frozen patient-independent splits
# ------------------------------------------------------------

train_mask_09 = (
    ml_data_09["split"] == "train"
)

val_mask_09 = (
    ml_data_09["split"] == "validation"
)

test_mask_09 = (
    ml_data_09["split"] == "test"
)

X_train_raw_09 = ml_data_09.loc[
    train_mask_09,
    FEATURE_COLUMNS_09
].to_numpy(dtype=np.float64)

X_val_raw_09 = ml_data_09.loc[
    val_mask_09,
    FEATURE_COLUMNS_09
].to_numpy(dtype=np.float64)

X_test_raw_09 = ml_data_09.loc[
    test_mask_09,
    FEATURE_COLUMNS_09
].to_numpy(dtype=np.float64)

Y_train_09 = ml_data_09.loc[
    train_mask_09,
    TARGET_COLUMNS_09
].to_numpy(dtype=np.int64)

Y_val_09 = ml_data_09.loc[
    val_mask_09,
    TARGET_COLUMNS_09
].to_numpy(dtype=np.int64)

Y_test_09 = ml_data_09.loc[
    test_mask_09,
    TARGET_COLUMNS_09
].to_numpy(dtype=np.int64)

print("\nRaw dataset shapes:")
print("  X_train:", X_train_raw_09.shape)
print("  X_validation:", X_val_raw_09.shape)
print("  X_test:", X_test_raw_09.shape)
print("  Y_train:", Y_train_09.shape)
print("  Y_validation:", Y_val_09.shape)
print("  Y_test:", Y_test_09.shape)

# ------------------------------------------------------------
# 8. Verify expected split sizes
# ------------------------------------------------------------

expected_sizes_09 = {
    "train": 17418,
    "validation": 2183,
    "test": 2198
}

actual_sizes_09 = {
    "train": int(train_mask_09.sum()),
    "validation": int(val_mask_09.sum()),
    "test": int(test_mask_09.sum())
}

if actual_sizes_09 != expected_sizes_09:
    raise RuntimeError(
        f"Unexpected split sizes:\n"
        f"{actual_sizes_09}"
    )

print("\nSplit sizes: PASS")

# ------------------------------------------------------------
# 9. Verify targets are binary
# ------------------------------------------------------------

for target_index, target_name in enumerate(
    TARGET_COLUMNS_09
):

    unique_values = np.unique(
        Y_train_09[:, target_index]
    )

    if not set(unique_values).issubset({0, 1}):
        raise RuntimeError(
            f"Non-binary target: {target_name}"
        )

print("Binary target encoding: PASS")

# ------------------------------------------------------------
# 10. Report training prevalence
# ------------------------------------------------------------

print("\nTraining target prevalence:")

for i, target in enumerate(TARGET_COLUMNS_09):

    positive_count = int(
        Y_train_09[:, i].sum()
    )

    prevalence = (
        positive_count /
        len(Y_train_09)
    ) * 100

    print(
        f"  {target:5s}: "
        f"{positive_count:5d} "
        f"({prevalence:6.2f}%)"
    )

# ------------------------------------------------------------
# 11. Scale using TRAINING DATA ONLY
# ------------------------------------------------------------

scaler_09 = StandardScaler()

X_train_09 = scaler_09.fit_transform(
    X_train_raw_09
)

X_val_09 = scaler_09.transform(
    X_val_raw_09
)

X_test_09 = scaler_09.transform(
    X_test_raw_09
)

print("\nFeature scaling:")
print("  Fit on training data only: PASS")
print("  Validation transformed: PASS")
print("  Test transformed: PASS")

# ------------------------------------------------------------
# 12. Verify scaled features
# ------------------------------------------------------------

if not np.isfinite(X_train_09).all():
    raise RuntimeError(
        "Non-finite scaled training features."
    )

if not np.isfinite(X_val_09).all():
    raise RuntimeError(
        "Non-finite scaled validation features."
    )

if not np.isfinite(X_test_09).all():
    raise RuntimeError(
        "Non-finite scaled test features."
    )

print("Scaled feature finiteness: PASS")

# ------------------------------------------------------------
# 13. Verify patient independence
# ------------------------------------------------------------

train_patients_09 = set(
    ml_data_09.loc[
        train_mask_09,
        "patient_id"
    ]
)

val_patients_09 = set(
    ml_data_09.loc[
        val_mask_09,
        "patient_id"
    ]
)

test_patients_09 = set(
    ml_data_09.loc[
        test_mask_09,
        "patient_id"
    ]
)

if train_patients_09 & val_patients_09:
    raise RuntimeError(
        "Patient overlap: train/validation."
    )

if train_patients_09 & test_patients_09:
    raise RuntimeError(
        "Patient overlap: train/test."
    )

if val_patients_09 & test_patients_09:
    raise RuntimeError(
        "Patient overlap: validation/test."
    )

print("\nPatient independence: PASS")

# ------------------------------------------------------------
# 14. Lock baseline protocol
# ------------------------------------------------------------

BASELINE_PROTOCOL_09 = {
    "targets": TARGET_COLUMNS_09,
    "feature_count": 120,
    "representation": "10 engineered features × 12 leads",
    "split": "patient-independent",
    "train_n": 17418,
    "validation_n": 2183,
    "test_n": 2198,
    "scaling": "StandardScaler fitted on training set only",
    "primary_metric": "macro_AUROC",
    "secondary_metrics": [
        "macro_AUPRC",
        "per_class_AUROC",
        "per_class_AUPRC",
        "per_class_F1",
        "per_class_sensitivity",
        "per_class_specificity"
    ],
    "test_usage": "final locked evaluation only"
}

print("\nLocked baseline protocol:")

for key, value in BASELINE_PROTOCOL_09.items():
    print(f"  {key}: {value}")

print("\n" + "=" * 70)
print("STEP 09F STATUS: PASS")
print("=" * 70)

print(
    "Frozen feature and target artifacts merged successfully."
)

print(
    "Five diagnostic targets locked:"
)

print(
    "  MI | STTC | CD | HYP | NORM"
)

print(
    "Training-only scaling applied."
)

print(
    "Patient-independent split verified."
)

print(
    "TEST SET HAS NOT BEEN USED FOR MODEL SELECTION."
)

STEP 09F-FIX — BASELINE ML DATASET CONSTRUCTION

Feature artifact:
(21799, 123)

Target artifact:
(21799, 8)

Source artifact integrity: PASS

Merged ML dataset:
(21799, 128)
21,799 ECGs × 120 features + 3 metadata + 5 targets: PASS
Feature count: 120 PASS
Target count: 5 PASS
Feature finiteness: PASS

Raw dataset shapes:
  X_train: (17418, 120)
  X_validation: (2183, 120)
  X_test: (2198, 120)
  Y_train: (17418, 5)
  Y_validation: (2183, 5)
  Y_test: (2198, 5)

Split sizes: PASS
Binary target encoding: PASS

Training target prevalence:
  MI   :  4379 ( 25.14%)
  STTC :  4186 ( 24.03%)
  CD   :  3907 ( 22.43%)
  HYP  :  2119 ( 12.17%)
  NORM :  7596 ( 43.61%)

Feature scaling:
  Fit on training data only: PASS
  Validation transformed: PASS
  Test transformed: PASS
Scaled feature finiteness: PASS

Patient independence: PASS

Locked baseline protocol:
  targets: ['MI', 'STTC', 'CD', 'HYP', 'NORM']
  feature_count: 120
  representation: 10 engineered features × 12 leads
  split: patient-

In [16]:
# ============================================================
# STEP 09G — MULTILABEL LOGISTIC REGRESSION BASELINE
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("STEP 09G — LOGISTIC REGRESSION BASELINE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Confirm required data exist
# ------------------------------------------------------------

required_objects_09g = [
    "X_train_09",
    "X_val_09",
    "Y_train_09",
    "Y_val_09",
    "TARGET_COLUMNS_09"
]

missing_09g = [
    name
    for name in required_objects_09g
    if name not in globals()
]

if missing_09g:
    raise RuntimeError(
        f"Missing required objects: {missing_09g}"
    )

print("\nRequired ML objects: PASS")

# ------------------------------------------------------------
# 2. Define the baseline model
# ------------------------------------------------------------

logistic_models_09 = {}

for target_index, target_name in enumerate(
    TARGET_COLUMNS_09
):

    print(
        f"\nTraining Logistic Regression: "
        f"{target_name}"
    )

    model = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=42
    )

    model.fit(
        X_train_09,
        Y_train_09[:, target_index]
    )

    logistic_models_09[target_name] = model

    print("  Training: PASS")

print("\nAll five models trained: PASS")

# ------------------------------------------------------------
# 3. Generate VALIDATION probabilities only
# ------------------------------------------------------------

Y_val_proba_09 = np.zeros(
    (
        len(X_val_09),
        len(TARGET_COLUMNS_09)
    ),
    dtype=np.float64
)

for target_index, target_name in enumerate(
    TARGET_COLUMNS_09
):

    model = logistic_models_09[target_name]

    Y_val_proba_09[:, target_index] = (
        model.predict_proba(
            X_val_09
        )[:, 1]
    )

# ------------------------------------------------------------
# 4. Validate probability matrix
# ------------------------------------------------------------

if Y_val_proba_09.shape != Y_val_09.shape:
    raise RuntimeError(
        f"Probability shape mismatch: "
        f"{Y_val_proba_09.shape} vs "
        f"{Y_val_09.shape}"
    )

if not np.isfinite(Y_val_proba_09).all():
    raise RuntimeError(
        "Non-finite validation probabilities detected."
    )

if (
    Y_val_proba_09.min() < 0
    or Y_val_proba_09.max() > 1
):
    raise RuntimeError(
        "Validation probabilities outside [0, 1]."
    )

print("\nValidation probability matrix: PASS")

# ------------------------------------------------------------
# 5. Calculate per-class AUROC
# ------------------------------------------------------------

val_auroc_09 = {}

for target_index, target_name in enumerate(
    TARGET_COLUMNS_09
):

    y_true = Y_val_09[:, target_index]
    y_score = Y_val_proba_09[:, target_index]

    auroc = roc_auc_score(
        y_true,
        y_score
    )

    val_auroc_09[target_name] = auroc

# ------------------------------------------------------------
# 6. Calculate macro-AUROC
# ------------------------------------------------------------

macro_auroc_09 = float(
    np.mean(
        list(val_auroc_09.values())
    )
)

print("\n" + "-" * 70)
print("VALIDATION AUROC")
print("-" * 70)

for target_name in TARGET_COLUMNS_09:

    print(
        f"{target_name:5s}: "
        f"{val_auroc_09[target_name]:.6f}"
    )

print(
    f"\nMACRO-AUROC: "
    f"{macro_auroc_09:.6f}"
)

# ------------------------------------------------------------
# 7. Create validation results table
# ------------------------------------------------------------

validation_results_09 = pd.DataFrame({
    "target": TARGET_COLUMNS_09,
    "AUROC": [
        val_auroc_09[target]
        for target in TARGET_COLUMNS_09
    ]
})

validation_results_09.loc[
    len(validation_results_09)
] = [
    "MACRO",
    macro_auroc_09
]

print("\nValidation results table:")
print(validation_results_09)

# ------------------------------------------------------------
# 8. Save validation baseline result
# ------------------------------------------------------------

BASELINE_RESULT_DIR = os.path.join(
    RESULTS_PATH,
    "baseline_ml"
)

os.makedirs(
    BASELINE_RESULT_DIR,
    exist_ok=True
)

VALIDATION_AUROC_PATH = os.path.join(
    BASELINE_RESULT_DIR,
    "logistic_regression_validation_auroc.csv"
)

validation_results_09.to_csv(
    VALIDATION_AUROC_PATH,
    index=False
)

print(
    "\nSaved validation result:"
)

print(
    VALIDATION_AUROC_PATH
)

# ------------------------------------------------------------
# 9. Save validation probabilities
# ------------------------------------------------------------

VAL_PROBA_PATH = os.path.join(
    BASELINE_RESULT_DIR,
    "logistic_regression_validation_probabilities.csv"
)

val_probability_output_09 = pd.DataFrame(
    Y_val_proba_09,
    columns=[
        f"{target}_probability"
        for target in TARGET_COLUMNS_09
    ]
)

val_probability_output_09.insert(
    0,
    "ecg_id",
    ml_data_09.loc[
        val_mask_09,
        "ecg_id"
    ].to_numpy()
)

val_probability_output_09.to_csv(
    VAL_PROBA_PATH,
    index=False
)

print(
    "Saved validation probabilities:"
)

print(
    VAL_PROBA_PATH
)

# ------------------------------------------------------------
# 10. Explicit test-set protection
# ------------------------------------------------------------

if "Y_test_proba_09" in globals():
    raise RuntimeError(
        "Unexpected test probability object detected."
    )

print("\nTEST SET EVALUATION: NOT PERFORMED")

print("\n" + "=" * 70)
print("STEP 09G STATUS: PASS")
print("=" * 70)

print(
    "Five independent binary Logistic Regression "
    "models trained for the multilabel targets."
)

print(
    "Validation AUROC calculated."
)

print(
    "Test set remains untouched."
)

STEP 09G — LOGISTIC REGRESSION BASELINE

Required ML objects: PASS

Training Logistic Regression: MI
  Training: PASS

Training Logistic Regression: STTC
  Training: PASS

Training Logistic Regression: CD
  Training: PASS

Training Logistic Regression: HYP
  Training: PASS

Training Logistic Regression: NORM
  Training: PASS

All five models trained: PASS

Validation probability matrix: PASS

----------------------------------------------------------------------
VALIDATION AUROC
----------------------------------------------------------------------
MI   : 0.823041
STTC : 0.871991
CD   : 0.846201
HYP  : 0.865227
NORM : 0.907121

MACRO-AUROC: 0.862716

Validation results table:
  target     AUROC
0     MI  0.823041
1   STTC  0.871991
2     CD  0.846201
3    HYP  0.865227
4   NORM  0.907121
5  MACRO  0.862716

Saved validation result:
/content/drive/MyDrive/PTB-XL Research Project/results/baseline_ml/logistic_regression_validation_auroc.csv
Saved validation probabilities:
/content/drive/M

In [17]:
# ============================================================
# STEP 09H — RANDOM FOREST BASELINE
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("STEP 09H — RANDOM FOREST BASELINE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

required_objects_09h = [
    "X_train_09",
    "X_val_09",
    "Y_train_09",
    "Y_val_09",
    "TARGET_COLUMNS_09"
]

missing_09h = [
    name
    for name in required_objects_09h
    if name not in globals()
]

if missing_09h:
    raise RuntimeError(
        f"Missing required objects: {missing_09h}"
    )

print("\nRequired ML objects: PASS")

# ------------------------------------------------------------
# 2. Train one Random Forest per diagnostic target
# ------------------------------------------------------------

random_forest_models_09 = {}

for target_index, target_name in enumerate(
    TARGET_COLUMNS_09
):

    print(
        f"\nTraining Random Forest: "
        f"{target_name}"
    )

    model = RandomForestClassifier(
        n_estimators=300,
        criterion="gini",
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_09,
        Y_train_09[:, target_index]
    )

    random_forest_models_09[target_name] = model

    print("  Training: PASS")

print("\nAll five Random Forest models trained: PASS")

# ------------------------------------------------------------
# 3. Generate VALIDATION probabilities only
# ------------------------------------------------------------

Y_val_proba_rf_09 = np.zeros(
    (
        len(X_val_09),
        len(TARGET_COLUMNS_09)
    ),
    dtype=np.float64
)

for target_index, target_name in enumerate(
    TARGET_COLUMNS_09
):

    model = random_forest_models_09[target_name]

    Y_val_proba_rf_09[:, target_index] = (
        model.predict_proba(
            X_val_09
        )[:, 1]
    )

# ------------------------------------------------------------
# 4. Validate probabilities
# ------------------------------------------------------------

if Y_val_proba_rf_09.shape != Y_val_09.shape:
    raise RuntimeError(
        f"Probability shape mismatch: "
        f"{Y_val_proba_rf_09.shape} vs "
        f"{Y_val_09.shape}"
    )

if not np.isfinite(Y_val_proba_rf_09).all():
    raise RuntimeError(
        "Non-finite validation probabilities detected."
    )

if (
    Y_val_proba_rf_09.min() < 0
    or Y_val_proba_rf_09.max() > 1
):
    raise RuntimeError(
        "Validation probabilities outside [0, 1]."
    )

print("\nValidation probability matrix: PASS")

# ------------------------------------------------------------
# 5. Calculate per-class AUROC
# ------------------------------------------------------------

val_auroc_rf_09 = {}

for target_index, target_name in enumerate(
    TARGET_COLUMNS_09
):

    auroc = roc_auc_score(
        Y_val_09[:, target_index],
        Y_val_proba_rf_09[:, target_index]
    )

    val_auroc_rf_09[target_name] = auroc

# ------------------------------------------------------------
# 6. Calculate macro-AUROC
# ------------------------------------------------------------

macro_auroc_rf_09 = float(
    np.mean(
        list(val_auroc_rf_09.values())
    )
)

print("\n" + "-" * 70)
print("RANDOM FOREST — VALIDATION AUROC")
print("-" * 70)

for target_name in TARGET_COLUMNS_09:

    print(
        f"{target_name:5s}: "
        f"{val_auroc_rf_09[target_name]:.6f}"
    )

print(
    f"\nMACRO-AUROC: "
    f"{macro_auroc_rf_09:.6f}"
)

# ------------------------------------------------------------
# 7. Compare directly with Logistic Regression
# ------------------------------------------------------------

if "val_auroc_09" not in globals():
    raise RuntimeError(
        "Logistic Regression validation results "
        "are not available."
    )

comparison_09 = pd.DataFrame({
    "target": TARGET_COLUMNS_09,
    "Logistic_Regression_AUROC": [
        val_auroc_09[target]
        for target in TARGET_COLUMNS_09
    ],
    "Random_Forest_AUROC": [
        val_auroc_rf_09[target]
        for target in TARGET_COLUMNS_09
    ]
})

comparison_09["RF_minus_Logistic"] = (
    comparison_09["Random_Forest_AUROC"]
    -
    comparison_09["Logistic_Regression_AUROC"]
)

comparison_09.loc[len(comparison_09)] = [
    "MACRO",
    macro_auroc_09,
    macro_auroc_rf_09,
    macro_auroc_rf_09 - macro_auroc_09
]

print("\n" + "-" * 70)
print("MODEL COMPARISON — VALIDATION")
print("-" * 70)

print(comparison_09)

# ------------------------------------------------------------
# 8. Save validation comparison
# ------------------------------------------------------------

BASELINE_RESULT_DIR = os.path.join(
    RESULTS_PATH,
    "baseline_ml"
)

os.makedirs(
    BASELINE_RESULT_DIR,
    exist_ok=True
)

RF_AUROC_PATH = os.path.join(
    BASELINE_RESULT_DIR,
    "random_forest_validation_auroc.csv"
)

comparison_09.to_csv(
    RF_AUROC_PATH,
    index=False
)

print(
    "\nSaved validation comparison:"
)

print(
    RF_AUROC_PATH
)

# ------------------------------------------------------------
# 9. Save Random Forest validation probabilities
# ------------------------------------------------------------

RF_PROBA_PATH = os.path.join(
    BASELINE_RESULT_DIR,
    "random_forest_validation_probabilities.csv"
)

rf_probability_output_09 = pd.DataFrame(
    Y_val_proba_rf_09,
    columns=[
        f"{target}_probability"
        for target in TARGET_COLUMNS_09
    ]
)

rf_probability_output_09.insert(
    0,
    "ecg_id",
    ml_data_09.loc[
        val_mask_09,
        "ecg_id"
    ].to_numpy()
)

rf_probability_output_09.to_csv(
    RF_PROBA_PATH,
    index=False
)

print(
    "Saved Random Forest validation probabilities:"
)

print(
    RF_PROBA_PATH
)

# ------------------------------------------------------------
# 10. Explicit test-set protection
# ------------------------------------------------------------

print("\nTEST SET EVALUATION: NOT PERFORMED")

print("\n" + "=" * 70)
print("STEP 09H STATUS: PASS")
print("=" * 70)

print(
    "Random Forest nonlinear baseline established."
)

print(
    "Validation-only comparison against Logistic Regression completed."
)

print(
    "Test set remains untouched."
)

STEP 09H — RANDOM FOREST BASELINE

Required ML objects: PASS

Training Random Forest: MI
  Training: PASS

Training Random Forest: STTC
  Training: PASS

Training Random Forest: CD
  Training: PASS

Training Random Forest: HYP
  Training: PASS

Training Random Forest: NORM
  Training: PASS

All five Random Forest models trained: PASS

Validation probability matrix: PASS

----------------------------------------------------------------------
RANDOM FOREST — VALIDATION AUROC
----------------------------------------------------------------------
MI   : 0.839991
STTC : 0.873967
CD   : 0.858054
HYP  : 0.887386
NORM : 0.912314

MACRO-AUROC: 0.874342

----------------------------------------------------------------------
MODEL COMPARISON — VALIDATION
----------------------------------------------------------------------
  target  Logistic_Regression_AUROC  Random_Forest_AUROC  RF_minus_Logistic
0     MI                   0.823041             0.839991           0.016951
1   STTC               

In [18]:
# ============================================================
# STEP 09I — FEATURE-BASED BASELINE ANALYSIS
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 09I — FEATURE-BASED BASELINE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Confirm required results
# ------------------------------------------------------------

required_09i = [
    "TARGET_COLUMNS_09",
    "val_auroc_09",
    "val_auroc_rf_09",
    "macro_auroc_09",
    "macro_auroc_rf_09"
]

missing_09i = [
    name for name in required_09i
    if name not in globals()
]

if missing_09i:
    raise RuntimeError(
        f"Missing required objects: {missing_09i}"
    )

print("\nRequired baseline results: PASS")

# ------------------------------------------------------------
# 2. Build comparison table
# ------------------------------------------------------------

baseline_comparison_09 = pd.DataFrame({
    "target": TARGET_COLUMNS_09,
    "Logistic_Regression_AUROC": [
        val_auroc_09[target]
        for target in TARGET_COLUMNS_09
    ],
    "Random_Forest_AUROC": [
        val_auroc_rf_09[target]
        for target in TARGET_COLUMNS_09
    ]
})

baseline_comparison_09["Absolute_Improvement"] = (
    baseline_comparison_09["Random_Forest_AUROC"]
    -
    baseline_comparison_09["Logistic_Regression_AUROC"]
)

baseline_comparison_09["Relative_Improvement_Percent"] = (
    baseline_comparison_09["Absolute_Improvement"]
    /
    baseline_comparison_09["Logistic_Regression_AUROC"]
    * 100
)

# ------------------------------------------------------------
# 3. Add macro-average row
# ------------------------------------------------------------

macro_row_09 = pd.DataFrame([{
    "target": "MACRO",
    "Logistic_Regression_AUROC": macro_auroc_09,
    "Random_Forest_AUROC": macro_auroc_rf_09,
    "Absolute_Improvement": (
        macro_auroc_rf_09 - macro_auroc_09
    ),
    "Relative_Improvement_Percent": (
        (macro_auroc_rf_09 - macro_auroc_09)
        /
        macro_auroc_09
        * 100
    )
}])

baseline_comparison_09 = pd.concat(
    [
        baseline_comparison_09,
        macro_row_09
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 4. Rank diagnostic categories by RF improvement
# ------------------------------------------------------------

ranked_improvement_09 = (
    baseline_comparison_09[
        baseline_comparison_09["target"] != "MACRO"
    ]
    .sort_values(
        "Absolute_Improvement",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked_improvement_09["Improvement_Rank"] = (
    np.arange(len(ranked_improvement_09)) + 1
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LOGISTIC REGRESSION VS RANDOM FOREST")
print("-" * 70)

display(
    baseline_comparison_09.round(6)
)

print("\n" + "-" * 70)
print("DIAGNOSTIC CATEGORIES RANKED BY RF IMPROVEMENT")
print("-" * 70)

display(
    ranked_improvement_09[
        [
            "Improvement_Rank",
            "target",
            "Absolute_Improvement",
            "Relative_Improvement_Percent"
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 6. Identify strongest and weakest nonlinear gains
# ------------------------------------------------------------

strongest_gain_09 = ranked_improvement_09.iloc[0]
weakest_gain_09 = ranked_improvement_09.iloc[-1]

print("\nStrongest RF improvement:")
print(
    f"  {strongest_gain_09['target']}: "
    f"+{strongest_gain_09['Absolute_Improvement']:.6f}"
)

print("\nSmallest RF improvement:")
print(
    f"  {weakest_gain_09['target']}: "
    f"+{weakest_gain_09['Absolute_Improvement']:.6f}"
)

# ------------------------------------------------------------
# 7. Determine whether RF improves every target
# ------------------------------------------------------------

all_targets_improved_09 = bool(
    (
        baseline_comparison_09.loc[
            baseline_comparison_09["target"] != "MACRO",
            "Absolute_Improvement"
        ] > 0
    ).all()
)

print(
    "\nRandom Forest improves every diagnostic target:",
    all_targets_improved_09
)

# ------------------------------------------------------------
# 8. Save publication-ready comparison
# ------------------------------------------------------------

BASELINE_RESULT_DIR = os.path.join(
    RESULTS_PATH,
    "baseline_ml"
)

os.makedirs(
    BASELINE_RESULT_DIR,
    exist_ok=True
)

COMPARISON_PATH_09I = os.path.join(
    BASELINE_RESULT_DIR,
    "feature_based_baseline_comparison_validation.csv"
)

baseline_comparison_09.to_csv(
    COMPARISON_PATH_09I,
    index=False
)

RANKING_PATH_09I = os.path.join(
    BASELINE_RESULT_DIR,
    "feature_based_baseline_improvement_ranking_validation.csv"
)

ranked_improvement_09.to_csv(
    RANKING_PATH_09I,
    index=False
)

print("\nSaved:")
print(COMPARISON_PATH_09I)
print(RANKING_PATH_09I)

# ------------------------------------------------------------
# 9. Explicit test protection
# ------------------------------------------------------------

print("\nTEST SET EVALUATION: NOT PERFORMED")

print("\n" + "=" * 70)
print("STEP 09I STATUS: PASS")
print("=" * 70)

print(
    "Feature-based baseline analysis completed."
)

print(
    "Classical ML branch remains validation-only."
)

print(
    "No model selection was performed using the test set."
)

STEP 09I — FEATURE-BASED BASELINE ANALYSIS

Required baseline results: PASS

----------------------------------------------------------------------
LOGISTIC REGRESSION VS RANDOM FOREST
----------------------------------------------------------------------


,target,Logistic_Regression_AUROC,Random_Forest_AUROC,Absolute_Improvement,Relative_Improvement_Percent
0,MI,0.823041,0.839991,0.016951,2.059522
1,STTC,0.871991,0.873967,0.001975,0.226515
2,CD,0.846201,0.858054,0.011853,1.400746
3,HYP,0.865227,0.887386,0.022158,2.560961
4,NORM,0.907121,0.912314,0.005193,0.572467
5,MACRO,0.862716,0.874342,0.011626,1.347607



----------------------------------------------------------------------
DIAGNOSTIC CATEGORIES RANKED BY RF IMPROVEMENT
----------------------------------------------------------------------


,Improvement_Rank,target,Absolute_Improvement,Relative_Improvement_Percent
0,1,HYP,0.022158,2.560961
1,2,MI,0.016951,2.059522
2,3,CD,0.011853,1.400746
3,4,NORM,0.005193,0.572467
4,5,STTC,0.001975,0.226515



Strongest RF improvement:
  HYP: +0.022158

Smallest RF improvement:
  STTC: +0.001975

Random Forest improves every diagnostic target: True

Saved:
/content/drive/MyDrive/PTB-XL Research Project/results/baseline_ml/feature_based_baseline_comparison_validation.csv
/content/drive/MyDrive/PTB-XL Research Project/results/baseline_ml/feature_based_baseline_improvement_ranking_validation.csv

TEST SET EVALUATION: NOT PERFORMED

STEP 09I STATUS: PASS
Feature-based baseline analysis completed.
Classical ML branch remains validation-only.
No model selection was performed using the test set.


In [19]:
# ============================================================
# NOTEBOOK 09 — FINAL SUMMARY
# ECG ML GENERALIZATION & ROBUSTNESS
# ============================================================

print("=" * 80)
print("NOTEBOOK 09 — FINAL SUMMARY")
print("=" * 80)

print("""
OBJECTIVE
---------
Establish a patient-independent feature-based machine-learning
baseline for five major PTB-XL diagnostic categories:

MI, STTC, CD, HYP, NORM


DATA
----
Dataset: PTB-XL v1.0.3
ECGs: 21,799
Leads: 12
Sampling frequency: 500 Hz
Duration: 10 seconds
Patients: patient-independent train/validation/test separation


FEATURE REPRESENTATION
----------------------
120 engineered ECG features:
10 features × 12 leads

Per-lead features:
- RMS
- Standard deviation
- Peak-to-peak amplitude
- Mean absolute amplitude
- Maximum absolute amplitude
- Range
- Skewness
- Kurtosis
- Dominant frequency
- Normalized spectral entropy


DATA SPLIT
----------
Training:   17,418 ECGs
Validation:  2,183 ECGs
Test:       2,198 ECGs

Patient overlap between train, validation, and test: ZERO

Feature scaling:
StandardScaler fitted on training data only.


MODELS
------
1. Logistic Regression
   - L2 regularization
   - C = 1.0
   - lbfgs solver
   - max_iter = 2000
   - random_state = 42

2. Random Forest
   - 300 trees
   - Gini criterion
   - sqrt feature sampling
   - min_samples_leaf = 2
   - balanced class weighting
   - random_state = 42


VALIDATION RESULTS
------------------
Logistic Regression macro-AUROC: 0.862716
Random Forest macro-AUROC:       0.874342
Improvement:                     +0.011626


PER-CLASS RESULTS
-----------------
MI:
  Logistic Regression = 0.823041
  Random Forest       = 0.839991

STTC:
  Logistic Regression = 0.871991
  Random Forest       = 0.873967

CD:
  Logistic Regression = 0.846201
  Random Forest       = 0.858054

HYP:
  Logistic Regression = 0.865227
  Random Forest       = 0.887386

NORM:
  Logistic Regression = 0.907121
  Random Forest       = 0.912314


MAIN FINDING
------------
Random Forest improved validation AUROC for all five diagnostic
categories relative to Logistic Regression.

The largest improvement occurred for HYP (+0.022158), followed
by MI (+0.016951) and CD (+0.011853).

The overall macro-AUROC increased by +0.011626.


INTERPRETATION
--------------
The engineered ECG representation contains substantial diagnostic
information, while nonlinear feature interactions provide additional
discriminative value beyond the linear baseline.

The feature-based classical ML branch is therefore established as
a reference baseline for comparison with raw-waveform deep learning.


TEST SET PROTECTION
-------------------
The test set was NOT used for:
- model selection
- hyperparameter tuning
- threshold selection
- performance comparison
- final reporting

All model comparisons in this notebook were performed using the
validation set only.


FROZEN ARTIFACTS
----------------
- ptbxl_patient_independent_split.csv
- ptbxl_diagnostic_targets.csv
- ptbxl_baseline_features_120.csv
- logistic_regression_validation_auroc.csv
- logistic_regression_validation_probabilities.csv
- random_forest_validation_auroc.csv
- random_forest_validation_probabilities.csv
- feature_based_baseline_comparison_validation.csv
- feature_based_baseline_improvement_ranking_validation.csv


NEXT STAGE
----------
Notebook 10:
Raw 12-lead ECG deep-learning generalization.

The next model branch will evaluate whether learning directly from
the 12-lead ECG waveform provides improved diagnostic generalization
relative to the engineered feature-based baselines established here.

NOTEBOOK 09 STATUS: COMPLETE / FROZEN
""")

print("=" * 80)
print("NOTEBOOK 09 COMPLETE — READY FOR VERSION CONTROL")
print("=" * 80)

NOTEBOOK 09 — FINAL SUMMARY

OBJECTIVE
---------
Establish a patient-independent feature-based machine-learning
baseline for five major PTB-XL diagnostic categories:

MI, STTC, CD, HYP, NORM


DATA
----
Dataset: PTB-XL v1.0.3
ECGs: 21,799
Leads: 12
Sampling frequency: 500 Hz
Duration: 10 seconds
Patients: patient-independent train/validation/test separation


FEATURE REPRESENTATION
----------------------
120 engineered ECG features:
10 features × 12 leads

Per-lead features:
- RMS
- Standard deviation
- Peak-to-peak amplitude
- Mean absolute amplitude
- Maximum absolute amplitude
- Range
- Skewness
- Kurtosis
- Dominant frequency
- Normalized spectral entropy


DATA SPLIT
----------
Training:   17,418 ECGs
Validation:  2,183 ECGs
Test:       2,198 ECGs

Patient overlap between train, validation, and test: ZERO

Feature scaling:
StandardScaler fitted on training data only.


MODELS
------
1. Logistic Regression
   - L2 regularization
   - C = 1.0
   - lbfgs solver
   - max_iter = 2000
 